# Connect to Cognite Data Fusion

Sample notebook that authenticates with the **Cognite Python SDK** using OIDC client credentials, then runs a few read-only checks.

Fill in the values in the next cell yourself. Nothing is read from `.env` or the process environment.

In [ ]:
# Fill these in with your CDF values.
CDF_PROJECT = "your-cdf-project"
CDF_CLUSTER = "westeurope-1"
CDF_CLIENT_ID = "your-client-id"
CDF_CLIENT_SECRET = "your-client-secret"
CDF_TENANT_ID = "your-azure-tenant-id"

# Leave as-is unless your tenant uses a different token endpoint.
CDF_TOKEN_URL = f"https://login.microsoftonline.com/{CDF_TENANT_ID}/oauth2/v2.0/token"

# Used only by the optional view lookup cell at the bottom.
INSTANCE_SPACE = "ac_action_item_management"
VIEW_EXTERNAL_ID = "Action"
VIEW_VERSION = "1"

## Create a `CogniteClient`

OIDC client-credentials flow against Azure AD.

In [ ]:
from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials

base_url = f"https://{CDF_CLUSTER}.cognitedata.com"

credentials = OAuthClientCredentials(
    token_url=CDF_TOKEN_URL,
    client_id=CDF_CLIENT_ID,
    client_secret=CDF_CLIENT_SECRET,
    scopes=[f"{base_url}/.default"],
)

client = CogniteClient(
    ClientConfig(
        client_name="redonline-cdf-ingest-notebook",
        project=CDF_PROJECT,
        base_url=base_url,
        credentials=credentials,
    )
)

print(f"Connected client: project={client.config.project} base_url={client.config.base_url}")

## Verify the connection

This will fail if credentials, cluster, project, or network access are wrong.

In [ ]:
token = client.config.credentials.authorization_header()[1]
print(f"Token acquired: {bool(token)} (length={len(token) if token else 0})")

status = client.iam.token.inspect()
print(f"Token subject: {status.subject}")
print(f"Projects visible: {[p.project_url_name for p in (status.projects or [])]}")

## Optional read-only probes

These only list metadata. Skip a cell if your service principal lacks that capability.

In [ ]:
print("RAW databases:")
for db in client.raw.databases.list(limit=10):
    print(f"  - {db.name}")

In [ ]:
print("Data modeling spaces:")
for space in client.data_modeling.spaces.list(limit=10):
    print(f"  - {space.space}")

In [ ]:
from cognite.client.data_classes.data_modeling import ViewId

try:
    views = client.data_modeling.views.retrieve(
        ViewId(space=INSTANCE_SPACE, external_id=VIEW_EXTERNAL_ID, version=VIEW_VERSION)
    )
    if views:
        view = views[0]
        print(f"Found view {view.external_id} v{view.version} in space {view.space}")
        print(f"Properties: {list((view.properties or {}).keys())}")
    else:
        print(
            f"No view named {VIEW_EXTERNAL_ID} in space {INSTANCE_SPACE}. "
            "Update INSTANCE_SPACE / VIEW_EXTERNAL_ID / VIEW_VERSION in the first code cell."
        )
except Exception as exc:
    print(f"View lookup skipped or failed: {type(exc).__name__}: {exc}")